In [12]:
# 필요한 라이브러리 import
import os
from dotenv import load_dotenv

# 환경변수 로드
load_dotenv()

True

## 첫번째 tool : 뉴스 요약 도구

### Tavilysearch

In [13]:
# Tool 생성

from langchain_tavily import TavilySearch

search_new = TavilySearch(
    max_results=3,
    topic="general",               # 또는 "news", "finance" 등
    include_answer=True,           # 답변 포함 여부
    include_raw_content=False,     # 원본 내용 포함 여부
    include_images=False,          # 이미지 포함 여부
    search_depth="advanced",          # "basic" 또는 "advanced"
    include_domains=[
        "https://news.naver.com/section/105",
        "https://news.daum.net/"
    ],
    exclude_domains=None            # 필요하면 제외 도메인 지정 가능
)

In [14]:
# Tool 실행: .invoke

result_new = search_new.invoke("최신 삼성전자 뉴스들 알려줘.") #-> "입력하는 텍스트 -> query"

### summarize_news

In [17]:
from langchain.tools import tool

@tool
def summarize_news(topic: str) -> str:
    """특정 주제의 최신 뉴스를 검색하고 요약합니다."""
    try:
        # 검색 쿼리 생성
        search_query = f"{topic} 최신 뉴스 한국"
        
        # 검색 실행
        result_new = search_new.invoke(search_query)
        
        if not result_new['results']:
            return f"'{topic}'에 대한 뉴스를 찾을 수 없습니다."
        
        # 결과 요약
        summary = f"'{topic}' 관련 최신 뉴스 요약:\n\n"
        for i, result in enumerate(result_new['results'][:3], 1):
            title = result.get('title', '제목 없음')
            content = result.get('content', '내용 없음')
            url = result.get('url', '')
            
            summary += f"{i}. {title}\n"
            summary += f"   {content[:100]}...\n"
            summary += f"   {url}\n\n"
        
        return summary
        
    except Exception as e:
        return f"뉴스 검색 중 오류가 발생했습니다: {str(e)}"

In [18]:
# AI agent가 사용할 tool -> 활성화: invoke
summarize_news_result = summarize_news.invoke({
    "topic":"AI"
})

## 두번째 도구 : 날씨 정보 도구

### TavilySearch

In [19]:
from langchain_tavily import TavilySearch

search_weather = TavilySearch(
    max_results=3,
    topic="general",               # 또는 "news", "finance" 등
    include_answer=True,           # 답변 포함 여부
    include_raw_content=False,     # 원본 내용 포함 여부
    include_images=False,          # 이미지 포함 여부
    search_depth="advanced",          # "basic" 또는 "advanced"
    include_domains=[
        "https://weather.daum.net/",
        "https://www.weatheri.co.kr/" 
    ],
    exclude_domains=None            # 필요하면 제외 도메인 지정 가능
)

### search_weather_info

In [20]:
@tool  
def search_weather_info(city: str, today:str) -> str:
    """특정 도시의 현재 날씨 정보를 검색합니다.
        city: 도시 이름 (예: 서울, 부산)
        today: 오늘 날짜 (예: 2026년 10월 16일)
    """
    try:
        search_query = f"{today} {city} 현재 날씨 기온"
        result_weather = search_weather.invoke(search_query)
        
        if not result_weather['answer'] or len(result_weather['results']) < 1:
            return f"'{city}'의 날씨 정보를 찾을 수 없습니다."
        
        # 첫 번째 결과에서 날씨 정보 추출
        weather_info = f"""
        {today} {city} 날씨 정보:
        {result_weather['answer']}

        출처: {result_weather['results'][0]['url']}
        """
        
        return weather_info
        
    except Exception as e:
        return f"날씨 정보 검색 중 오류가 발생했습니다: {str(e)}"

In [15]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5-nano",
    reasoning_effort="high",        # 논리성 강화
)

## 세번째 도구 : 주식 정보 도구

### TavilySearch

In [21]:
from langchain_tavily import TavilySearch

search_finance = TavilySearch(
    max_results=3,
    topic="finance",               # 또는 "news", "finance" 등
    include_answer=True,           # 답변 포함 여부
    include_raw_content=False,     # 원본 내용 포함 여부
    include_images=False,          # 이미지 포함 여부
    search_depth="advanced",          # "basic" 또는 "advanced"
    include_domains=[
        "https://kr.investing.com/equities/south-korea",
        "https://finance.naver.com/"
    ],
    exclude_domains=None            # 필요하면 제외 도메인 지정 가능
)

### search_stock_info

In [22]:
@tool
def search_stock_info(stock_name: str, today:str) -> str:
    """특정 주식의 현재 가격과 정보를 검색합니다.
    - stock name: 주식명 (삼성전자, LG 전자)
    - today: 오늘 날짜 (2026sus 10월 16일)
    
    """
    try:
        search_query = f"{today} {stock_name} 주식 현재가 주가"
        result_finance = search_finance.invoke(search_query)

        if not result_finance['answer'] or len(result_finance['results']) < 1:
            return f"'{today} {stock_name}' 주식 정보를 찾을 수 없습니다."
        # 
        
        # 첫 번째 결과에서 날씨 정보 추출
        stock_info = f"""
        # {today} {stock_name} 주식 정보:
        {result_finance['results'][0]['title']}
        {result_finance['results'][0]['content']}

        출처: {result_finance['results'][0]['url']}
        """
        return stock_info
        
    except Exception as e:
        return f"주식 정보 검색 중 오류가 발생했습니다: {str(e)}"

## llm과 agent 설정

### LLM

In [24]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5-nano",
    reasoning_effort="high",        # 논리성 강화
)

### tools

In [25]:
# Agent가 사용할 tool list

tools = [summarize_news, search_weather_info, search_stock_info]

### Agent

In [26]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    당신은 주어진 도구를 반드시 사용해서만 답변해야 하는 AI 어시스턴트입니다.
    절대로 자신의 지식으로 직접 답변하지 마세요.
    모든 최종 답변은 반드시 도구의 출력 결과를 기반으로 해야 합니다.
    """
)

## 테스트

### 뉴스 검색

In [27]:
from langchain_core.messages import HumanMessage

user_input = "삼성전자 관련 최신 뉴스를 요약해주세요."
result1 = agent.invoke(
    {"messages": HumanMessage(content=user_input)}
)

In [28]:
print(result1['messages'][-1].content)

다음은 삼성전자 관련 최신 뉴스 요약입니다 (툴 출력 기준 요약):

- 코스피 반등의 지속 여부가 삼성전자와 SK하이닉스의 실적 및 전망에 달려 있다는 관점이 주목받고 있습니다. 투자자들이 이들 반도체 대기업의 실적에 주된 관심을 집중하고 있습니다. 
- 삼성전자의 호실적이 지수 상승의 지지 요인으로 언급됩니다. 삼성전자의 양호한 실적이 코스피 지수의 긍정적 흐름에 기여할 가능성이 제시되고 있습니다.

참고: 피드에는 삼성전자 관련 내용이 주를 이루지만, 비관련 뉴스도 함께 제공되고 있습니다. 원하시면 자세한 기사 원문 링크와 구체 수치들을 함께 정리해 드리겠습니다.


### 날씨 검색

In [29]:
from langchain_core.messages import HumanMessage

user_input = "서울의 현재 날씨를 알려주세요."
result2 = agent.invoke(
    {"messages": HumanMessage(content=user_input)}
)

In [30]:
print(result2['messages'][-1].content)

다음은 2026-10-18일 서울의 현재 날씨에 대한 수집 정보입니다. 두 출처 간 차이가 있어 확정은 어렵습니다.

- 소스 A: 기온 23.7°C, 부분적으로 구름 많음, 북동풍 1.5 m/s
- 소스 B: 기온 30.1°C, 10월 기록상 최고 기온으로 보도

출처: weather.daum.net

현재 온도에 대해 단정적으로 말하기 어렵다는 점 참고해 주세요. 더 정확한 실시간 정보를 원하시면 다른 신뢰 소스의 최신 데이터를 추가로 확인해 드리겠습니다. 어떤 소스를 더 확인해 볼까요?


### LangSmith에서 확인